# Model Feature Generation
Loads the base processed reviews, generates model-driven features (sentiment, emotion, topics), and saves a single enriched file for downstream analysis.

In [1]:
!pip install bertopic umap-learn sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.7 MB/s eta 0:00:00


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

2026-03-29 16:11:52.171323: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774800712.401868      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774800712.466399      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774800712.974755      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800712.974799      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774800712.974802      55 computation_placer.cc:177] computation placer alr

In [3]:
# --------------------------------------------
# Config
# --------------------------------------------
# INPUT_FILE = os.getenv("INPUT_FILE", "Processed_Reviews.csv")  # single source; load and save to same file
BATCH_SENTIMENT = 32
BATCH_EMOTION = 32
BATCH_EMBEDDINGS = 256
MAX_TEXT_LEN = 256  # tokenizer-level truncation; adjust if model context differs
TEXT_TRUNCATE_CHARS = None  # optional pre-truncation for very long texts; keep None to let tokenizer handle
CHUNK_CHARS_SENTIMENT = None  # set to an int (e.g., 2000) to chunk very long texts before sentiment model
CHUNK_CHARS_EMOTION = None    # set to an int (e.g., 2000) to chunk very long texts before emotion model
TOPIC_SAMPLE_N = None  # set to an int (e.g., 3000) to test topics on a subset first
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIPELINE_DEVICE = 0 if DEVICE == "cuda" else -1
print(f"Using device for embeddings: {DEVICE.upper()} | pipelines: {'GPU' if PIPELINE_DEVICE == 0 else 'CPU'}")

Using device for embeddings: CUDA | pipelines: GPU


In [4]:
# --------------------------------------------
# 1. Load base dataset
# --------------------------------------------
INPUT_FILE="/kaggle/input/datasets/anusankrishnathas/processedreviews1/Processed_Reviews.csv"
df = pd.read_csv(INPUT_FILE)
print(f"Loaded: {INPUT_FILE}")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

required_cols = [
    "Title", "Text", "Rating_Class",
    "Rating", "Location_Name", "Located_City", "Location_Type",
    "Province", "District", "User_Country", "User_Region",
    "Travel_Date_Month", "Travel_Date_Year",
    "Published_Date_Month", "Published_Date_Year",
    "User_Contributions", "Helpful_Votes",
    "Review_Length", "Title_Length", "Review_Delay_Days"
]
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in base file: {missing_cols}")

Loaded: /kaggle/input/datasets/anusankrishnathas/processedreviews1/Processed_Reviews.csv
Shape: (16156, 22)
Columns: ['Unnamed: 0', 'Location_Name', 'Located_City', 'Province', 'District', 'Location_Type', 'User_Locale', 'User_Country', 'User_Region', 'Travel_Date_Month', 'Travel_Date_Year', 'Published_Date_Month', 'Published_Date_Year', 'User_Contributions', 'Rating', 'Helpful_Votes', 'Title', 'Text', 'Review_Length', 'Title_Length', 'Rating_Class', 'Review_Delay_Days']


In [5]:
# --------------------------------------------
# 3. Helper functions
# --------------------------------------------
CARDIFF_LABEL_MAP = {"LABEL_0": "NEGATIVE", "LABEL_1": "NEUTRAL", "LABEL_2": "POSITIVE"}
SENTIMENT_NUMERIC_MAP = {"NEGATIVE": -1, "NEUTRAL": 0, "POSITIVE": 1}


def clean_and_truncate(text, truncate_chars=None):
    """Normalize text: strip, optional hard truncation for very long reviews."""
    if pd.isna(text):
        return ""
    cleaned = str(text).strip()
    if truncate_chars and len(cleaned) > truncate_chars:
        return cleaned[:truncate_chars]
    return cleaned


def chunk_text(text, chunk_chars=None):
    """Optionally split very long text into manageable chunks; returns list with at least one item."""
    if not chunk_chars or not text or len(text) <= chunk_chars:
        return [text]
    return [text[i:i + chunk_chars] for i in range(0, len(text), chunk_chars)]


def prepare_text_list(series, truncate_chars=None):
    """Prepare list for model input with consistent cleaning and optional truncation."""
    return [clean_and_truncate(t, truncate_chars) for t in series]


def normalize_sentiment_label(label):
    """Map raw/model labels to NEGATIVE/NEUTRAL/POSITIVE; unknowns become None."""
    if label is None or (isinstance(label, float) and pd.isna(label)):
        return None
    upper = str(label).strip().upper()
    if upper in CARDIFF_LABEL_MAP:
        return CARDIFF_LABEL_MAP[upper]
    if upper in SENTIMENT_NUMERIC_MAP:
        return upper
    return None


def normalize_emotion_label(label):
    """Normalize emotion labels to Title case; unknowns become None."""
    if label is None or (isinstance(label, float) and pd.isna(label)):
        return None
    cleaned = str(label).strip().replace("_", " ")
    return cleaned.title() if cleaned else None


def map_label_to_numeric_series(series):
    """Vectorized sentiment-to-numeric mapping with prior normalization."""
    normalized = series.apply(normalize_sentiment_label)
    return normalized.map(SENTIMENT_NUMERIC_MAP)


def run_model_in_batches(texts, pipe, batch_size=32, max_len=256, desc="", chunk_chars=None):
    """Batch inference with optional chunking; skips empty texts but preserves alignment.
    Chunking aggregates per-review predictions by taking the highest-score chunk (keeps one label/score per row)."""
    n = len(texts)
    labels = [None] * n
    scores = [0.0] * n
    tasks = []  # (row_idx, text_chunk)
    for idx, text in enumerate(texts):
        if not text:
            continue
        for chunk in chunk_text(text, chunk_chars=chunk_chars):
            tasks.append((idx, chunk))
    total = len(tasks)
    for start in tqdm(range(0, total, batch_size), total=(total + batch_size - 1) // batch_size, desc=desc or "Batches"):
        batch_slice = tasks[start:start + batch_size]
        batch_indices = [idx for idx, _ in batch_slice]
        batch_texts = [txt for _, txt in batch_slice]
        try:
            outputs = pipe(batch_texts, truncation=True, max_length=max_len, batch_size=batch_size)
        except Exception as exc:
            print(f"Batch {start} failed: {exc}")
            continue
        for idx, out in zip(batch_indices, outputs):
            score = float(out.get("score", 0.0))
            label = out.get("label")
            if labels[idx] is None or score > scores[idx]:
                labels[idx] = label
                scores[idx] = round(score, 4)
    return labels, scores


def embed_texts(texts, model, batch_size=256, device="cpu"):
    """Precompute sentence embeddings once for BERTopic."""
    safe_texts = [t if t else "" for t in texts]
    return model.encode(
        safe_texts,
        batch_size=batch_size,
        show_progress_bar=True,
        device=device,
        normalize_embeddings=False,
    )


def get_topic_keywords(topic_id, topic_model, top_n=5):
    if topic_id == -1:
        return "Outlier"
    topic_words = topic_model.get_topic(topic_id)
    if not topic_words:
        return None
    return ", ".join([f"{word}({round(score, 2)})" for word, score in topic_words[:top_n]])


def max_topic_probability(prob):
    """Safely extract the max topic probability from BERTopic output (array or scalar)."""
    if prob is None:
        return None
    try:
        arr = np.asarray(prob, dtype=float)
        if arr.size == 0:
            return None
        return float(np.nanmax(arr))
    except Exception:
        return None


def assign_themes(topics_series, topic_theme_map):
    """Vectorized theme mapping with safe fallback to 'Other'."""
    return topics_series.map(topic_theme_map).fillna("Other")

In [6]:
# --------------------------------------------
# 2. Create Combined_Text
# --------------------------------------------
df["Title"] = df["Title"].fillna("").astype(str)
df["Text"] = df["Text"].fillna("").astype(str)
df["Combined_Text"] = (df["Title"] + " " + df["Text"]).str.strip()
df["Combined_Text"] = df["Combined_Text"].replace("", np.nan)
print("Non-empty Combined_Text rows:", df["Combined_Text"].notna().sum())


# --------------------------------------------
# 4. Centralized text preparation
# --------------------------------------------
combined_text_list = prepare_text_list(
    df["Combined_Text"],
    truncate_chars=TEXT_TRUNCATE_CHARS,
 )
print("Prepared texts for modeling:", len(combined_text_list))

Non-empty Combined_Text rows: 16156
Prepared texts for modeling: 16156


In [7]:
# --------------------------------------------
# 4. Load models
# --------------------------------------------
print("Loading models...")
sentiment_pipe = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=PIPELINE_DEVICE,
 )
emotion_pipe = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    device=PIPELINE_DEVICE,
 )
embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
topic_model = BERTopic(
    embedding_model=None,  # embeddings are precomputed to avoid double work
    calculate_probabilities=True,
    umap_model=UMAP(n_neighbors=10, n_components=5, low_memory=True, random_state=42),
    vectorizer_model=CountVectorizer(stop_words="english"),
    verbose=True,
 )
print("Models ready.")

Loading models...


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models ready.


In [8]:
# --------------------------------------------
# 6. Sentiment prediction (batched)
# --------------------------------------------
print("Running sentiment model...")
sent_labels, sent_scores = run_model_in_batches(
    combined_text_list,
    sentiment_pipe,
    batch_size=BATCH_SENTIMENT,
    max_len=MAX_TEXT_LEN,
    desc="Sentiment",
    chunk_chars=CHUNK_CHARS_SENTIMENT,
 )
df["Combined_Sentiment"] = pd.Series(sent_labels).apply(normalize_sentiment_label)
df["Sentiment_Score"] = sent_scores
print(df[["Combined_Sentiment", "Sentiment_Score"]].head())

Running sentiment model...


Sentiment:   0%|          | 0/505 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Combined_Sentiment  Sentiment_Score
0           POSITIVE           0.9782
1           POSITIVE           0.9883
2           POSITIVE           0.9885
3           POSITIVE           0.9781
4           POSITIVE           0.9816


In [ ]:
# --------------------------------------------
# 7. Emotion prediction (batched)
# --------------------------------------------
print("Running emotion model...")
emotion_labels, _ = run_model_in_batches(
    combined_text_list,
    emotion_pipe,
    batch_size=BATCH_EMOTION,
    max_len=MAX_TEXT_LEN,
    desc="Emotion",
    chunk_chars=CHUNK_CHARS_EMOTION,
 )
df["Emotion"] = [normalize_emotion_label(x) for x in emotion_labels]
print(df[["Emotion"]].head())

Running emotion model...


Emotion:   0%|          | 0/505 [00:00<?, ?it/s]

In [10]:
# --------------------------------------------
# 8. Topic modeling with BERTopic
# --------------------------------------------
print("Running BERTopic (this may take a while)...")
topic_texts = pd.Series(combined_text_list).fillna("").astype(str).str.strip()
if TOPIC_SAMPLE_N is not None:
    topic_texts = topic_texts.iloc[:TOPIC_SAMPLE_N]
    df_subset = df.iloc[: len(topic_texts)].copy()
else:
    df_subset = df
topic_text_list = topic_texts.tolist()
embeddings = embed_texts(topic_text_list, embedding_model, batch_size=BATCH_EMBEDDINGS, device=DEVICE)
topics, probs = topic_model.fit_transform(topic_text_list, embeddings=embeddings)
df_subset["Dominant_Topic"] = topics
if probs is not None:
    df_subset["Topic_Probability"] = [max_topic_probability(p) for p in probs]
else:
    df_subset["Topic_Probability"] = None
topic_keywords = [get_topic_keywords(t, topic_model, top_n=5) for t in df_subset["Dominant_Topic"]]
df_subset["Topic_Keywords"] = topic_keywords
topic_info = topic_model.get_topic_info()
print(topic_info.head(15))
if TOPIC_SAMPLE_N is not None:
    df.update(df_subset[["Dominant_Topic", "Topic_Probability", "Topic_Keywords"]])
else:
    df = df_subset

2026-03-29 16:19:42,898 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-29 16:20:13,965 - BERTopic - Dimensionality - Completed ✓
2026-03-29 16:20:13,966 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-29 16:20:29,835 - BERTopic - Cluster - Completed ✓
2026-03-29 16:20:29,844 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-29 16:20:30,627 - BERTopic - Representation - Completed ✓


    Topic  Count                                Name  \
0      -1   3896      -1_elephants_temple_park_place   
1       0   1290          0_tea_factory_tour_process   
2       1   1030            1_beach_clean_waves_sand   
3       2    355  2_stupa_dagoba_anuradhapura_stupas   
4       3    333           3_tea_factory_ceylon_tour   
5       4    289   4_falls_waterfall_fall_waterfalls   
6       5    279               5_lake_boat_walk_fish   
7       6    278     6_tsunami_moving_photos_graphic   
8       7    264        7_gardens_garden_trees_house   
9       8    249     8_museum_history_lankan_culture   
10      9    227         9_temple_hindu_inside_shoes   
11     10    196     10_seat_liptons_haputale_lipton   
12     11    195          11_beach_beaches_sri_lanka   
13     12    189        12_fort_walls_dutch_ramparts   
14     13    187    13_spices_spice_products_massage   

                                                                                   Representation  \
0 

In [11]:
# --------------------------------------------
# 11. Post-fit theme mapping (inspect then map)
# --------------------------------------------
# Inspect topic_info / Topic_Keywords above, then adjust this mapping if IDs shift. BERTopic topic IDs are not stable across reruns.
topic_theme_map = {
    0: "Scenery",
    1: "Cultural Experience",
    2: "Service Quality",
    3: "Crowding & Pricing",
    4: "Food & Hospitality",
    5: "Wildlife & Nature",
    # Add/update after reviewing topic_info to keep themes aligned with topics.
}
# For any unmapped topic IDs, default to "Other" and log them for review.
df["Review_Theme"] = assign_themes(df["Dominant_Topic"], topic_theme_map)
unmapped_topics = sorted(set(df["Dominant_Topic"]) - set(topic_theme_map))
if unmapped_topics:
    print("[Warning] Unmapped topic IDs detected (defaulted to 'Other'):", unmapped_topics)
    sample_keywords = {tid: get_topic_keywords(tid, topic_model, top_n=5) for tid in unmapped_topics}
    print("Sample keywords for unmapped topics:", sample_keywords)
print(df[["Dominant_Topic", "Review_Theme"]].head())

[Warning] Unmapped topic IDs detected (defaulted to 'Other'): [-1, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151]
Sample keywords for unmapped topics: {-1: 'Outlier', 6: 'tsunami(0.08), moving(0.03), photos(0.03), graphic(0.03), 2004(0.03)', 7: 'gardens(0.06), garden(0.05), trees(0.03), house(0.03), flowers(0.03)', 8: 'museum(0.04), history(0.03), lankan(0.02), culture(0.02), sri(0.02)', 9: 'temple(0.05), hindu(

In [12]:
# --------------------------------------------
# 9. Sentiment-rating gap
# --------------------------------------------
rating_numeric = map_label_to_numeric_series(df["Rating_Class"])
sentiment_numeric = map_label_to_numeric_series(df["Combined_Sentiment"])
valid_mask = rating_numeric.notna() & sentiment_numeric.notna()
df["Sentiment_Rating_Gap"] = np.nan
df.loc[valid_mask, "Sentiment_Rating_Gap"] = (rating_numeric[valid_mask] - sentiment_numeric[valid_mask]).abs()
print(df[["Rating_Class", "Combined_Sentiment", "Sentiment_Rating_Gap"]].head())

  Rating_Class Combined_Sentiment  Sentiment_Rating_Gap
0     Positive           POSITIVE                   0.0
1     Positive           POSITIVE                   0.0
2     Positive           POSITIVE                   0.0
3     Positive           POSITIVE                   0.0
4     Positive           POSITIVE                   0.0


In [13]:
# --------------------------------------------
# 9b. Ensure all column segments start capitalized
# --------------------------------------------
def _capitalize_segments(name: str) -> str:
    return "_".join([part[:1].upper() + part[1:] if part else part for part in name.split("_")])

df.rename(columns={c: _capitalize_segments(c) for c in df.columns}, inplace=True)

In [15]:
# --------------------------------------------
# 10. Save enriched dataset (overwrite input)
# --------------------------------------------
# Keep raw text columns available during processing; drop only at export to reduce file size.
OUTPUT_FILE = "/kaggle/working/Processed_Reviews.csv"

print("Saved:", OUTPUT_FILE)
df = df.drop(columns=["Unnamed: 0", "Text", "Title"], errors="ignore")
df.to_csv(OUTPUT_FILE, index=False)
# print(f"Saved: {INPUT_FILE}")
print("Final shape:", df.shape)
print("Columns:")
for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

Saved: /kaggle/working/Processed_Reviews.csv
Final shape: (16156, 28)
Columns:
 1. Location_Name
 2. Located_City
 3. Province
 4. District
 5. Location_Type
 6. User_Locale
 7. User_Country
 8. User_Region
 9. Travel_Date_Month
10. Travel_Date_Year
11. Published_Date_Month
12. Published_Date_Year
13. User_Contributions
14. Rating
15. Helpful_Votes
16. Review_Length
17. Title_Length
18. Rating_Class
19. Review_Delay_Days
20. Combined_Text
21. Combined_Sentiment
22. Sentiment_Score
23. Emotion
24. Dominant_Topic
25. Topic_Probability
26. Topic_Keywords
27. Review_Theme
28. Sentiment_Rating_Gap
